# 🎬 YouTube Content Creator AI Agent (Google Colab Edition)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thakoralpesh3/youtube-agent/blob/main/YouTube_Agent_Colab.ipynb)

Autonomous AI Agent that generates, edits, voices, subtitles, and publishes **YouTube Shorts** (Trilingual: English, Hindi, Gujarati) and **Standard 16:9 Masterclass Videos** directly to YouTube.

---

### 🚀 Quick Start in 3 Easy Steps:
1. **Run Steps 1 & 2** to clone the repository and install system dependencies (FFmpeg + fonts).
2. **Run Steps 3 & 4** to install Python packages and configure your credentials (`GEMINI_API_KEY`, `client_secret.json`, `token.json`).
3. **Run Steps 5 & 6** to test your YouTube channel connection and generate/publish your videos!


In [ ]:
# @title 📥 1. Clone & Setup Repository
import os
import sys
from pathlib import Path

REPO_URL = 'https://github.com/thakoralpesh3/youtube-agent.git'
WORK_DIR = Path('/content/youtube-agent')

if not WORK_DIR.exists():
    print(f'Cloning {REPO_URL} into {WORK_DIR}...')
    !git clone {REPO_URL} {WORK_DIR}
else:
    print(f'Updating existing repository at {WORK_DIR}...')
    %cd {WORK_DIR}
    !git pull origin main

%cd {WORK_DIR}
print(f'\n✅ Working directory ready: {os.getcwd()}')


In [ ]:
# @title ⚙️ 2. Install System Dependencies & Fonts
print('Installing FFmpeg and multi-language fonts (Noto, CJK, Indic, Gujarati, Hindi)...')
!apt-get update -qq > /dev/null
!apt-get install -y -qq ffmpeg fonts-noto fonts-noto-cjk fonts-noto-extra fonts-indic > /dev/null
print('✅ System packages and fonts installed successfully!')


In [ ]:
# @title 📦 3. Install Python Dependencies
print('Installing Python packages from requirements.txt...')
!pip install -q -r requirements.txt
print('✅ Python dependencies installed successfully!')


In [ ]:
# @title 🔐 4. Configure Credentials & Secrets
# Provide credentials via ONE of the following methods:
# Method A (Recommended): Colab Secrets (Click Key 🔑 icon on left sidebar)
#           Names: GEMINI_API_KEY, YOUTUBE_CLIENT_SECRET_JSON, YOUTUBE_TOKEN_JSON
# Method B: Direct File Upload (Check 'upload_files_manually' below)
# Method C: Google Drive Mount (Check 'mount_google_drive' below)

upload_files_manually = False  # @param {type:'boolean'}
mount_google_drive = False     # @param {type:'boolean'}

import os
import json
from pathlib import Path

# Method A: Colab Secrets
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    if gemini_key:
        with open('.env', 'w') as f:
            f.write(f'GEMINI_API_KEY={gemini_key}\n')
            f.write('DISPATCH_TO_GITHUB_ACTIONS=false\n')
            f.write('YOUTUBE_PRIVACY_STATUS=public\n')
        print('✅ Loaded GEMINI_API_KEY from Colab Secrets.')
    
    yt_client = userdata.get('YOUTUBE_CLIENT_SECRET_JSON')
    if yt_client:
        with open('client_secret.json', 'w') as f:
            f.write(yt_client if isinstance(yt_client, str) else json.dumps(yt_client))
        print('✅ Loaded client_secret.json from Colab Secrets.')

    yt_token = userdata.get('YOUTUBE_TOKEN_JSON')
    if yt_token:
        with open('token.json', 'w') as f:
            f.write(yt_token if isinstance(yt_token, str) else json.dumps(yt_token))
        print('✅ Loaded token.json from Colab Secrets.')
except Exception:
    pass

# Method C: Google Drive Mount
if mount_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_secrets = Path('/content/drive/MyDrive/youtube-agent-secrets')
    if drive_secrets.exists():
        for fname in ['.env', 'client_secret.json', 'token.json', 'social_credentials.json']:
            src = drive_secrets / fname
            if src.exists():
                with open(fname, 'wb') as dst:
                    dst.write(src.read_bytes())
                print(f'✅ Loaded {fname} from Google Drive.')

# Method B: Direct File Upload Widget
if upload_files_manually or (not Path('.env').exists() and not Path('token.json').exists()):
    from google.colab import files
    print('\n📁 Please upload credentials (.env, client_secret.json, token.json):\n')
    uploaded = files.upload()
    for fname, content in uploaded.items():
        with open(fname, 'wb') as f:
            f.write(content)
        print(f'✅ Saved uploaded file: {fname}')

print('\n--- Verification Status ---')
env_ok = '✅ Ready' if Path('.env').exists() else '❌ Missing'
client_ok = '✅ Ready' if Path('client_secret.json').exists() else '❌ Missing'
token_ok = '✅ Ready' if Path('token.json').exists() else '❌ Missing'
print(f'🔑 .env               : {env_ok}')
print(f'🔐 client_secret.json : {client_ok}')
print(f'🎫 token.json         : {token_ok}')


In [ ]:
# @title 🔍 5. Verify YouTube Channel Connection & Diagnostics
import os
import json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv('.env')

# 1. Verify Gemini API Connection
gemini_key = os.getenv('GEMINI_API_KEY', '')
if gemini_key:
    try:
        from google import genai
        client = genai.Client(api_key=gemini_key)
        models = [m.name.replace('models/', '') for m in client.models.list() if 'generateContent' in (getattr(m, 'supported_actions', []) or [])]
        print(f'✅ Gemini API Connected! Available models: {len(models)}')
        print('   Recommended active model: gemini-3-flash-preview')
    except Exception as e:
        print(f'⚠️ Gemini API Notice: {e}')
else:
    print('❌ GEMINI_API_KEY is not set.')

# 2. Verify YouTube OAuth Channel
token_path = Path('token.json')
if token_path.exists():
    try:
        from google.oauth2.credentials import Credentials
        from googleapiclient.discovery import build
        token_data = json.loads(token_path.read_text(encoding='utf-8'))
        creds = Credentials.from_authorized_user_info(token_data)
        yt = build('youtube', 'v3', credentials=creds)
        res = yt.channels().list(mine=True, part='snippet,statistics').execute()
        if res.get('items'):
            ch = res['items'][0]
            title = ch['snippet']['title']
            subs = ch['statistics'].get('subscriberCount', 'N/A')
            vids = ch['statistics'].get('videoCount', 'N/A')
            print(f'\n🎉 YouTube Channel Connected: "{title}"')
            print(f'   📊 Subscribers: {subs} | Total Published Videos: {vids}')
        else:
            print('⚠️ No channel returned for this token.')
    except Exception as e:
        print(f'⚠️ YouTube API Notice: {e}')
else:
    print('❌ token.json not found. Run Step 4 to provide credentials.')


In [ ]:
# @title 🚀 6. Run Autonomous YouTube Creator Agent
# Configure execution parameters:
content_mode = 'Trilingual Shorts + 16:9 Long Video' # @param ['Trilingual Shorts + 16:9 Long Video', 'Trilingual Shorts Only', 'Standard 16:9 Video Only']
custom_topic = '' # @param {type:'string'}
gemini_model = 'gemini-3-flash-preview' # @param ['gemini-3-flash-preview', 'gemini-3.8-flash', 'gemini-4-argon', 'gemini-3.1-pro-preview']
dry_run = False # @param {type:'boolean'}
privacy_status = 'public' # @param ['public', 'unlisted', 'private']

import os
os.environ['YOUTUBE_PRIVACY_STATUS'] = privacy_status
os.environ['GEMINI_MODEL'] = gemini_model
os.environ['DISPATCH_TO_GITHUB_ACTIONS'] = 'false'

from agent import YouTubeAgent
agent = YouTubeAgent(model=gemini_model)

topic = custom_topic.strip() if custom_topic.strip() else None

if content_mode == 'Trilingual Shorts + 16:9 Long Video':
    print(f'🎬 Running Trilingual Shorts + Standard 16:9 Masterclass Video (Topic: {topic or "Auto-Trend"})...')
    result = agent.run_trilingual(
        topic=topic,
        target_platform='youtube',
        dry_run=dry_run,
        force=True,
    )
elif content_mode == 'Trilingual Shorts Only':
    print(f'📱 Running Trilingual Shorts Only (Topic: {topic or "Auto-Trend"})...')
    result = agent.run_trilingual(
        topic=topic,
        target_platform='youtube',
        dry_run=dry_run,
        force=True,
    )
else:
    print(f'🎥 Running Standard 16:9 Video Only (Topic: {topic or "Auto-Trend"})...')
    result = agent.generate_standard_video(
        topic=topic,
        dry_run=dry_run,
    )

print('\n🎉 Autonomous Content Generation Finished!')


In [ ]:
# @title 📺 7. Preview Generated Videos Inline
from pathlib import Path
from IPython.display import HTML, display

output_dir = Path('output')
mp4_files = sorted(list(output_dir.glob('run_*/**/*.mp4')), key=lambda p: p.stat().st_mtime, reverse=True)

if mp4_files:
    print(f'Found {len(mp4_files)} generated video(s). Showing top results:\n')
    for p in mp4_files[:3]:
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f'• {p.name} ({size_mb:.2f} MB) -> {p}')
else:
    print('ℹ️ No generated videos found in output/ yet. Run Step 6 to generate content.')


In [ ]:
# @title 💾 8. Download Generated Videos & Package to Local Computer
from pathlib import Path
import shutil
from google.colab import files

output_dir = Path('output')
latest_runs = sorted(list(output_dir.glob('run_*')), key=lambda p: p.stat().st_mtime, reverse=True)

if latest_runs:
    latest_run = latest_runs[0]
    zip_name = f'{latest_run.name}_videos'
    print(f'Archiving {latest_run} -> {zip_name}.zip...')
    archive_path = shutil.make_archive(zip_name, 'zip', str(latest_run))
    print(f'✅ Archive created: {archive_path} ({Path(archive_path).stat().st_size / (1024*1024):.2f} MB)')
    print('Triggering browser download...')
    files.download(archive_path)
else:
    print('ℹ️ No run folders found in output/ to download.')
